# 03 - YOLO 架构详解**学习目标：**- 理解 YOLO 的三段式结构：Backbone → Neck → Head- 理解特征金字塔（FPN/PAN）的多尺度检测原理- 学会查看和解读模型结构- 理解 anchor-free 机制---

## 1. 整体架构YOLOv8/v11 的架构可以分为三部分：```输入图片 (640×640×3)       │       ▼┌─────────────┐│   Backbone   │  特征提取：从像素中提取语义特征│   (C2f)     │  类似 ResNet 的作用└──────┬──────┘       │  多尺度特征图       ▼┌─────────────┐│    Neck      │  特征融合：聚合不同尺度的特征│  (FPN+PAN)  │  让小目标和大目标都能被检测└──────┬──────┘       │  融合后的特征图       ▼┌─────────────┐│    Head      │  检测头：从特征生成预测│ (Decoupled) │  输出 bounding box + 类别└──────┬──────┘       │       ▼  NMS 后处理 → 最终检测结果```

## 2. Backbone：特征提取**作用**：把原始像素转换成有语义含义的特征向量。**演化历史**：```Darknet-53 (YOLOv3) → CSPDarknet (YOLOv4/v5) → C2f (YOLOv8/v11)```**C2f (Cross Stage Partial with 2 convolutions)**：- 拆分特征通道，一部分做卷积，一部分直接传递- 减少计算量，同时保持梯度流- 比传统的 ResBlock 更高效**输出**：3 个不同尺度的特征图```P3: 80×80×256   ← 浅层，保留细节，检测小目标P4: 40×40×512   ← 中层P5: 20×20×1024  ← 深层，语义丰富，检测大目标```

## 3. Neck：特征融合**问题**：深层特征语义强但空间信息弱，浅层特征空间信息强但语义弱。**解决方案**：FPN + PAN 双向融合```P5 ──────┬─────────────────┬────── P5_out         │ ↓ top-down      │ ↑ bottom-upP4 ──────┼─────────────────┼────── P4_out         │ ↓ top-down      │ ↑ bottom-upP3 ──────┴─────────────────┴────── P3_out         FPN (自上而下)      PAN (自下而上)```- **FPN** (Feature Pyramid Network)：深层→浅层，传递语义信息- **PAN** (Path Aggregation Network)：浅层→深层，传递定位信息- 融合后的每个尺度都有丰富的语义+空间信息

## 4. Head：检测头**YOLOv8 使用 Decoupled Head**（解耦头）：```特征图  │  ├──→ 分支1: Classification → 类别概率 (nc 个)  │  └──→ 分支2: Regression → 边界框坐标 (4 个)         + DFL (Distribution Focal Loss)```**Anchor-based vs Anchor-free**：| 方式 | 描述 | YOLO 版本 ||------|------|-----------|| Anchor-based | 预定义参考框，预测偏移量 | v3/v4/v5 || **Anchor-free** | 直接预测框坐标 | **v8/v11** |Anchor-free 的优势：- 不需要预设 anchor 尺寸- 更灵活，适应不同形状的目标- 减少超参数

## 5. 动手实验：查看模型结构

In [ ]:
from ultralytics import YOLO# 加载 YOLO11 nano 模型model = YOLO("yolo11n.pt")# 查看模型结构摘要print("模型结构摘要：")model.info(verbose=True)

In [ ]:
# 查看模型的层级结构print("模型层级：")print(type(model.model))print()# 查看各层参数量total_params = sum(p.numel() for p in model.model.parameters())trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)print(f"总参数量: {total_params:,}")print(f"可训练参数: {trainable_params:,}")

In [ ]:
# 查看不同大小模型的对比
from ultralytics import YOLO

print("YOLO11 各版本对比：")
print(f"{'版本':<10} {'参数量':<15} {'文件大小'}")
print("-" * 40)

for size in ['n', 's', 'm']:
    m = YOLO(f"yolo11{size}.pt")
    params = sum(p.numel() for p in m.model.parameters())
    print(f"yolo11{size:<5} {params:>12,}    {size}")

## 6. 多尺度检测原理为什么需要多尺度？因为目标大小差异很大：```┌─────────────────────────────────┐│ 🚗  🚗                         │  ← 远处的车很小 (20×20 px)│           ┌───────────┐         ││           │           │         ││           │  🚛 卡车   │         │  ← 近处的车很大 (200×200 px)│           │           │         ││           └───────────┘         │└─────────────────────────────────┘```**检测头分配**：| 特征图 | 尺寸 | 感受野 | 检测目标 ||--------|------|--------|----------|| P3 | 80×80 | 小 | 小目标 (人、远处的车) || P4 | 40×40 | 中 | 中等目标 (车、动物) || P5 | 20×20 | 大 | 大目标 (建筑、卡车) |每个尺度有独立的检测头，最终合并所有结果。

## 7. 练习### 练习 1: 对比不同模型大小加载 yolo11n, yolo11s, yolo11m，用同一张图片推理，对比：- 推理速度- 检测结果### 练习 2: 查看特征图使用 `model.model` 访问内部层，尝试提取中间特征图。### 练习 3: 参数量分析统计 Backbone、Neck、Head 各自的参数量占比。

## 📝 学习检查

加载本章检查点和练习，检验学习效果：

```python
from yolo_learn.pedagogy.checkpoint import Quiz
from yolo_learn.pedagogy.scaffold import ExerciseSet
from yolo_learn.pedagogy.reflection import ReflectionSet, MistakeLibrary
```


In [ ]:
# 加载本章检查点
quiz = Quiz.from_yaml("03_yolo_architecture")
print(f"本章 {quiz.total} 个检查点 + 预测练习")

# 查看第一题
print(quiz.ask(0))


### ✍️ 练习


In [ ]:
# 加载本章练习
exercises = ExerciseSet.from_yaml("03_yolo_architecture")
print(f"本章 {exercises.total} 个练习")
for ex in exercises.exercises:
    print()
    print(ex.show())


### 🤔 反思与自评


In [ ]:
# 加载反思问题和自评量表
reflections = ReflectionSet.from_yaml("03_yolo_architecture")
print(reflections.show_reflections())
print()
print(reflections.show_assessments())

# 常见错误
mistakes = MistakeLibrary()
print()
print(mistakes.show_for_topic("yolo_architecture"))


---**上一节**: [02 - 数据集与标注](02_dataset_and_annotations.ipynb)**下一节**: [04 - 推理与预测](04_inference_with_pretrained.ipynb) - 用预训练模型做推理